# SQL MURDER CITY

The problem can be found at: https://mystery.knightlab.com/

- A crime has taken place and the detective needs your help
- The detective gave you the crime scene report, but you somehow lost it
- You vaguely remember that the crime was a ​murder​ that occurred sometime on ​Jan.15, 2018​ and that it took place in ​SQL City
- Start by retrieving the corresponding crime scene report from the police department’s database

In [ ]:
-- Find more information regarding the murder by reading the crime report
SELECT * 
FROM Crime_scene_report
WHERE 
    type = 'murder' 
    AND city = 'SQL City' 
    AND date = '20180115'            

| date | type | description | city |
|---|---|---|---|
| 20180115 | murder | Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave". | SQL City |

In [ ]:
-- Find more information about the witnesses (note that ORDER BY and LIMIT would have been applied to the same SELECT statement as UNION so we have to separate them)
SELECT * 
FROM (
  	SELECT * 
  	FROM Person
	WHERE address_street_name = 'Northwestern Dr'
	ORDER BY address_number DESC
	LIMIT 1
)

UNION 

SELECT * 
FROM Person 
WHERE 
	address_street_name = 'Franklin Ave'
	AND name LIKE 'Annabel%'

| id | name | license_id | address_number | address_street_name | ssn |
|---|---|---:|---:|---|---:|
| 14887 | Morty Schapiro | 118009 | 4919 | Northwestern Dr | 111564949 |
| 16371 | Annabel Miller | 490173 | 103 | Franklin Ave | 318771143 |

In [ ]:
-- Have these people been interviewed? If so, what have they reported?
SELECT 
	p.id,
	p.name,
	i.transcript
FROM Interview i
LEFT JOIN Person p ON i.person_id = p.id
WHERE p.id = 14887 OR p.id = 16371

| id | name | transcript |
|---|---|---|
| 14887 | Morty Schapiro | I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W". |
| 16371 | Annabel Miller | I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th. |

In [ ]:
/* Important information discovered: 
- Suspect seem a "man"
- Probably goes to the "Get Fit Now Gym" and has a gold memebership
- Bag membership number on bag starts with "48Z"
- Suspect got into a car with "H42W" contained in the plate

- The suspect has been recognized going to the same gym as Annabel
- The suspect was at the gym on January 9th 2018
*/

-- Gold membership at the gym on the reported day by Annabel
SELECT 
	g.membership_id, g.check_in_date,
	m.person_id, m.name, m.membership_status
FROM Get_fit_now_check_in g
LEFT JOIN Get_fit_now_member m ON g.membership_id = m.id
WHERE 
	g.check_in_date = '20180109' 
	AND m.membership_status = 'gold'
	AND m.id LIKE '48Z%'

| membership_id | check_in_date | person_id | name | membership_status |
|---|---|---|---|---|
| 48Z7A | 20180109 | 28819 | Joe Germuska | gold |
| 48Z55 | 20180109 | 67318 | Jeremy Bowers | gold |

In [ ]:
-- People associated to cars with plate matching the reported piece
SELECT 
	d.id, d.age, d.plate_number, d.car_make, d.car_model,
	p.name
FROM Drivers_license d
LEFT JOIN Person p ON d.id = p.license_id
WHERE 
	d.plate_number LIKE 'H42W%'
	OR d.plate_number LIKE '%H42W'
	OR d.plate_number LIKE '%H42W%'

| id | age | plate_number | car_make | car_model | name |
|---|---|---|---|---|---|
| 183779 | 21 | H42W0X | Toyota | Prius | Maxine Whitely |
| 423327 | 30 | 0H42W2 | Chevrolet | Spark LS | Jeremy Bowers |
| 664760 | 21 | 4H42WR | Nissan | Altima | Tushar Chandra |

In [ ]:
-- The murder is Jeremy Bowers given it's gold membership status, matching membership ID, matching car license plate, and checked in the gym on Jan. 9th 2018
INSERT INTO solution 
VALUES (1, 'Jeremy Bowers');
        
SELECT value FROM solution;

In [ ]:
-- Let's see if Jeremy Bowers has releasted any interview statement
SELECT p.id, p.name, i.transcript
FROM Person p
LEFT JOIN Interview i ON p.id = i.person_id
WHERE i.person_id = 67318

| id | name | transcript |
|---|---|---|
| 67318 | Jeremy Bowers | I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5" (65") or 5'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017. |

In [ ]:
-- There seems to be another suspect to identify, who actually hired the killer
-- Since the killer is uncertain on the height we filter by all other criteria


| id | name | annual_income | age | height | hair_color | gender | car_make | car_model |
|---:|---|---:|---:|---:|---|---|---|---|
| 99716 | Miranda Priestly | 310000 | 68 | 66 | red | female | Tesla | Model S |
| 90700 | Regina George | null | 65 | 66 | red | female | Tesla | Model S |
| 78881 | Red Korb | 278000 | 48 | 65 | red | female | Tesla | Model S |

In [ ]:
-- Need to find who attended the SQL symphony city concerts
SELECT 
	p.id, p.name, 
	f.event_name, f.date
FROM Facebook_event_checkin f
LEFT JOIN Person p ON f.person_id = p.id
WHERE 
	f.person_id IN (78881, 90700, 99716)
	AND (f.date > 20171130 AND f.date < 20180101)

| id | name | event_name | date |
|---:|---|---|---:|
| 99716 | Miranda Priestly | SQL Symphony Concert | 20171206 |
| 99716 | Miranda Priestly | SQL Symphony Concert | 20171212 |
| 99716 | Miranda Priestly | SQL Symphony Concert | 20171229 |

In [ ]:
-- We found the person who hired the killer!
INSERT INTO solution 
VALUES (1, 'Miranda Priestly');

SELECT value FROM solution;